# 02 · Covariate extraction (versioned)
The label-free covariates come from SRTM, MERIT Hydro, CHIRPS, Sentinel-2, TerraClimate, OpenStreetMap,
the Copernicus GLO-30 DEM and USGS ShakeMap (`src/gee_extraction.py`, `src/extract_hydro_catchment_gee.py`,
`src/extract_osm_roads.py`, `src/hydro_gis.py`, `src/seismic.py`). Re-extracting the Earth Engine layers
needs an authenticated account, so the extraction outputs are versioned under `data/processed/` and this
notebook verifies them instead of re-downloading: coverage, ranges and the tier definitions used
everywhere else. Every other notebook runs from these files alone.

In [1]:
import logging, os, sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", stream=sys.stdout, force=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

M = ROOT / "results/metrics"
F = ROOT / "results/figures"

def table(name, **kw):
    return pd.read_csv(M / name, **kw).round(4)

def fig(name, width=1100):
    display(Image(filename=str(F / name), width=width))

In [2]:
from src.dataset import load_master
from src.features import FEATURE_SETS

df = load_master()
for name, cols in FEATURE_SETS.items():
    missing = [c for c in cols if c not in df.columns]
    print(f"{name:18s} {len(cols):3d} covariates   missing from master: {missing or 'none'}")

env_hydro_seismic   45 covariates   missing from master: none
env                 23 covariates   missing from master: none
env_season          24 covariates   missing from master: none
env_hydro           39 covariates   missing from master: none
env_rich            67 covariates   missing from master: none
env_hydro_season    40 covariates   missing from master: none
leaky_reference     26 covariates   missing from master: none


## Coverage and ranges of the extracted covariates

In [3]:
cols = FEATURE_SETS["env_rich"]
cov = df[cols].describe().T[["count", "mean", "std", "min", "max"]]
cov["share_missing"] = 1 - cov["count"] / len(df)
cov.round(3).sort_values("share_missing", ascending=False).head(20)

,count,mean,std,min,max,share_missing
soil_clay,5577.0,247.438,22.285,173.333,320.000,0.020
soil_bdod,5577.0,118.137,5.427,100.667,132.667,0.020
soil_soc,5577.0,244.493,63.061,160.333,579.333,0.020
soil_sand,5577.0,326.296,26.258,214.667,423.000,0.020
et_trend,5598.0,5.290,3.118,-3.268,26.317,0.016
et_mean,5598.0,701.936,117.418,520.774,1377.626,0.016
latitude,5689.0,27.567,0.066,27.395,27.744,0.000
spring_density_500m,5689.0,15.484,13.780,0.000,97.000,0.000
longitude,5689.0,85.610,0.090,85.412,85.819,0.000
elevation_m,5689.0,1385.821,326.678,542.000,2758.000,0.000


## The DEM-routing and seismic layers used only in association analyses

In [4]:
hg = pd.read_csv(ROOT / "data/processed/hydro_gis_features.csv")
print(f"{len(hg):,} sources with DEM-routing features; "
      f"{hg.catch_T2p0.nunique()} occupied 2 km^2 link catchments")
hg.describe().T[["mean", "min", "max"]].round(3)

5,689 sources with DEM-routing features; 176 occupied 2 km^2 link catchments


,mean,min,max
id,2845.000,1.000,5689.000
dem_elev_m,1379.119,538.302,2750.151
contrib_area_km2,0.236,0.002,95.815
catch_T0p5,1380.408,289.000,2347.000
flowlen_to_stream_T0p5_m,470.979,0.000,2048.332
strahler_T0p5,1.575,1.000,6.000
catch_T2p0,307.062,65.000,529.000
flowlen_to_stream_T2p0_m,992.456,0.000,3660.810
strahler_T2p0,1.634,1.000,5.000
catch_T10p0,65.480,13.000,111.000
